In [ ]:
# HW07: Attention, BERTopic and DeepLatent

Remember that these homework work as a completion grade. **You can skip one section of this homework.**

## Attention

In [2]:
import math
import torch
import tiktoken

text = "Your journey starts with the start"

#TODO Tokenize the sentence with tiktoken (GPT-2)
# Load the GPT-2 BPE tokenizer from tiktoken
enc = tiktoken.get_encoding("gpt2")
# encode() maps the raw string to a list of integer token IDs
token_ids = enc.encode(text)
# Decode each individual ID back to its human-readable string piece
tokens = [enc.decode([tid]) for tid in token_ids]

print("Token IDs:", token_ids)
print("Tokens:", tokens)

Token IDs: [7120, 7002, 4940, 351, 262, 923]
Tokens: ['Your', ' journey', ' starts', ' with', ' the', ' start']


In [4]:
import numpy as np
import os
import zipfile
import urllib.request

# Download GloVe 6B if not already present (file is ~822 MB zipped)
if not os.path.exists('glove.6B.50d.txt'):
    print("Downloading GloVe 6B vectors...")
    urllib.request.urlretrieve(
        "https://nlp.stanford.edu/data/glove.6B.zip",
        "glove.6B.zip"
    )
    # Extract only the 50-dimensional file to save space
    with zipfile.ZipFile("glove.6B.zip", "r") as z:
        z.extract("glove.6B.50d.txt")
    os.remove("glove.6B.zip")
    print("Done.")

glove = {}
with open(r'glove.6B.50d.txt', 'rb') as f:
    for line in f:
        parts = line.split()
        word = parts[0].decode('utf-8')
        vector = np.array(parts[1:], dtype=np.float32)
        glove[word] = vector

#TODO Convert each token into a GloVe vector. If a token is not found in GloVe, assign a zero vector
embedding_dim = 50
token_vectors = []
# Tiktoken tokens may include a leading space (e.g. " starts"); strip and lowercase before lookup
for token in tokens:
    word = token.strip().lower()
    if word in glove:
        token_vectors.append(glove[word])
    else:
        # Out-of-vocabulary token — fall back to a zero vector of the correct dimension
        token_vectors.append(np.zeros(embedding_dim))

#TODO Stack into the input embedding matrix X
# np.stack converts the list of 1-D vectors into a 2-D matrix of shape (n_tokens, embedding_dim)
X = np.stack(token_vectors)
print("Tokens:", tokens)
print("Embedding matrix X shape:", X.shape)  # expected: (n_tokens, 50)

Done.
Tokens: ['Your', ' journey', ' starts', ' with', ' the', ' start']
Embedding matrix X shape: (6, 50)


In [5]:
#TODO use the CausalAttention class to generate the context vectors

import torch.nn as nn

class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length,
                dropout, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)            
        self.register_buffer(
           'mask',
           torch.triu(torch.ones(context_length, context_length),
           diagonal=1)
        )            

    def forward(self, x):
        b, num_tokens, d_in = x.shape                   
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.transpose(1, 2)   
        attn_scores.masked_fill_(                    
            self.mask.bool()[:num_tokens, :num_tokens], -torch.inf) 
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )
        attn_weights = self.dropout(attn_weights)

        context_vec = attn_weights @ values
        return context_vec

# Convert the numpy matrix X to a PyTorch float tensor
# unsqueeze(0) adds a batch dimension: shape becomes (1, n_tokens, d_in)
X_tensor = torch.tensor(X, dtype=torch.float32).unsqueeze(0)

d_in = embedding_dim   # 50, matching the GloVe vector size
d_out = 16             # output dimension for each attention head
context_length = X_tensor.shape[1]  # number of tokens in the sentence

# Create the causal attention module; dropout=0.0 so weights are fully retained at inference
torch.manual_seed(123)
ca = CausalAttention(d_in=d_in, d_out=d_out, context_length=context_length, dropout=0.0)

# torch.no_grad() disables gradient computation — we're only doing a forward pass
with torch.no_grad():
    context_vecs = ca(X_tensor)

# Output shape: (batch=1, n_tokens, d_out)
# Each row is a context-aware representation of that token informed by all previous tokens
print("Context vectors shape:", context_vecs.shape)
print("Context vectors:\n", context_vecs)

Context vectors shape: torch.Size([1, 6, 16])
Context vectors:
 tensor([[[-0.6016, -0.3429, -0.1523,  0.1515,  0.1972,  0.3499, -0.1046,
          -0.3759,  0.1129, -0.6095, -0.7215, -0.0846, -0.1252, -0.4704,
          -0.8412,  0.3113],
         [-0.4246, -0.4334,  0.0572,  0.1261,  0.1809,  0.3415, -0.0253,
          -0.5019,  0.2085, -0.3610, -0.7141, -0.1867, -0.2113, -0.2729,
          -0.5824,  0.1002],
         [-0.4407, -0.2548,  0.0593,  0.1106,  0.1065,  0.2202,  0.0580,
          -0.4862, -0.0027, -0.4651, -0.6412, -0.2670, -0.1759, -0.1643,
          -0.4809,  0.0846],
         [-0.4078, -0.2173, -0.0567,  0.1737,  0.1016,  0.0948,  0.1648,
          -0.3684, -0.1795, -0.4820, -0.5290, -0.3301, -0.1327, -0.1725,
          -0.5525,  0.1295],
         [-0.4313, -0.2224, -0.0787,  0.1182,  0.0783,  0.0872,  0.1966,
          -0.3191, -0.1914, -0.5440, -0.5164, -0.3821, -0.1189, -0.1700,
          -0.6019,  0.1652],
         [-0.4833, -0.1340, -0.1074,  0.1473,  0.0424,  0.126

## BERTopic on Congressional speeches

In [2]:
# Load a random sample of speeches pronounced in the floor of the US Congress between 1994 and 2024
import pandas as pd
df = pd.read_csv('us_congress_speeches_sample.csv') # Found on the same Github
df['year'] = pd.to_datetime(df['date']).dt.year

print("Number of speeches: {}".format(len(df)))
df.head()

Number of speeches: 28731


,date,text,speaker_bioguide,chamber,party,doc_clean,year
0,2003-06-10,"Mr. ISRAEL. Mr. Speaker, I rise today to com...",I000057,House,Democrat,commend induction successful wrestling coach h...,2003
1,2012-05-08,"Mr. LANGEVIN. Mr. Speaker, Democrats are com...",L000559,House,Democrat,commit reduce deficit balanced way contrast br...,2012
2,2010-03-25,"Mr. BISHOP of Utah. Mr. Speaker, Lori Garver...",B001250,House,Republican,number administrator political give credit pri...,2010
3,1995-01-25,"Mr. KIM. Mr. Chairman, I rise today in suppo...",K000181,House,Republican,balanced budget small business tough run small...,1995
4,2003-05-22,"Mr. BAUCUS. Mr. President, sometime in the n...",B000243,Senate,Democrat,near future eye beholder go conference tax cut...,2003


In [ ]:
import os
# Must be set before any numba/UMAP/HDBSCAN imports — macOS libomp crashes when
# nested OpenMP threading is triggered by numba alongside sentence-transformers
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

#!pip install bertopic[visualization]
from bertopic import BERTopic

#TODO fit a topic model (BERTopic) restricting to 10 topics
# doc_clean contains pre-processed text (stopwords removed, key terms kept)
docs = df['doc_clean'].tolist()

# nr_topics=11 because BERTopic always reserves topic -1 for outliers and counts
# it toward the target, so 11 yields 10 real topics + 1 outlier bucket
topic_model = BERTopic(nr_topics=11)
topics, probs = topic_model.fit_transform(docs)

print(topic_model.get_topic_info())

In [ ]:
# ---- COLAB VERSION (run this cell instead of the one above when using Google Colab) ----
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from bertopic import BERTopic
from umap import UMAP
from sentence_transformers import SentenceTransformer
import torch

docs = df['doc_clean'].tolist()

# Pre-compute embeddings on CPU in small batches — avoids VRAM competition between
# sentence-transformers, UMAP, and HDBSCAN all running on the T4 at the same time
embedding_model = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")
embeddings = embedding_model.encode(docs, batch_size=32, show_progress_bar=True)

# Free any GPU memory before UMAP/HDBSCAN take over
torch.cuda.empty_cache()

# low_memory=True switches UMAP from exact pairwise distance matrix (O(n²) RAM)
# to approximate nearest-neighbor search — prevents the session crash on 28k docs
umap_model = UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    low_memory=True,
    random_state=42
)

# nr_topics=11 so that after BERTopic reserves topic -1 for outliers we get 10 real topics
topic_model = BERTopic(nr_topics=11, umap_model=umap_model)
topics, probs = topic_model.fit_transform(docs, embeddings=embeddings)

print(topic_model.get_topic_info())

In [4]:
#TODO print out and visualize the top 5 words for each topic.

# get_topic_info() returns a DataFrame with topic IDs; topic -1 is BERTopic's outlier bucket
for topic_id in sorted(topic_model.get_topic_info()['Topic'].tolist()):
    if topic_id == -1:
        continue  # skip the outlier topic — these are docs that didn't fit any cluster
    words = topic_model.get_topic(topic_id)
    print(f"Topic {topic_id}: {[w for w, _ in words[:5]]}")

# Built-in bar chart showing the top-5 discriminative terms for each of the 10 topics
topic_model.visualize_barchart(top_n_topics=10, n_words=5)

Topic 0: ['bill', 'go', 'other', 'get', 'more']
Topic 1: ['honor', 'community', 'service', 'life', 'recognize']
Topic 2: ['bill', 'tax', 'budget', 'go', 'business']
Topic 3: ['bill', 'pass', 'move', 'other', 'read']
Topic 4: ['no', 'unable', 'roll', 'detain', 'record']
Topic 5: ['debt', 'stand', 'cent', 'dollar', 'federal']
Topic 6: ['friend', 'chairman', 'good', 'appreciate', 'like']
Topic 7: ['degree', 'resolution', 'second', 'offer', 'no']
Topic 8: ['reprint', 'date', 'lie', 'expense', 'list']


# DeepLatent

Now we will estimate a topic model but accounting for covariates (metadata)

In [12]:
!pip install deeplatent

import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from deeplatent.corpus import Corpus
from deeplatent.models import GTM

# Re-use the same cleaned text column as the document corpus
docs = df['doc_clean'].tolist()

vectorizer = CountVectorizer(
    ngram_range=(1, 1),
    max_df=0.5,    # drop words appearing in >50% of speeches — these generic fillers
                   # ("way", "thing", "community", "country") otherwise dominate every topic
    min_df=0.001,
    stop_words="english",
    token_pattern=r"(?u)\b\w{3,}\b"  # only words with 3 or more letters
)

# Fit the vocabulary on the full corpus so the same word index is used throughout
vectorizer.fit(docs)

print("Number of words in the vocabulary: {}".format(len(vectorizer.get_feature_names_out())))

# Define modalities: one text modality represented as a bag-of-words view.
# NOTE: the column must match the text we vectorized above -> "doc_clean"
modalities = {
    "text": {
        "column": "doc_clean",
        "views": {
            "bow": {
                "type": "bow",
                "vectorizer": vectorizer  # reuse the fitted vectorizer
            }
        }
    }
}

#TODO build the train dataset to estimate a GTM using as covariates the party, year and chamber for both prevalence and content.
# Covariates use R-style patsy formulas (same syntax as the lab's "~ C(year)").
# C(...) marks a column as categorical. We include party, year and chamber.
#   prevalence -> controls HOW MUCH each topic appears in a document
#   content    -> controls WHICH WORDS are used within a topic
train_dataset = Corpus(
    df=df,
    modalities=modalities,
    prevalence="~ C(party) + C(year) + C(chamber)",  #TODO add prevalence covs
    content="~ C(party) + C(year) + C(chamber)"      #TODO add content covs
)

#TODO provide a brief intuition on why prevalence might matter by year and content might matter by party
# Prevalence by year: the salience of political topics shifts across time — healthcare, war, and
# fiscal policy rise and fall with elections and world events, so modeling year captures that drift
# in HOW MUCH each topic is discussed.
# Content by party: Republicans and Democrats often discuss the *same* topic using different
# vocabulary (e.g., "tax relief" vs. "tax cuts for the wealthy"), so allowing the WORD distribution
# to vary by party surfaces ideological language differences within each topic.

Number of words in the vocabulary: 6038


In [13]:
#TODO estimate a GTM (same specs as in the lab). Retrain to the amount of steps that you think necessary for the topics to make sense. Use 10 topics.

# Same architecture as the lab: two-layer MLP encoder/decoder over the bag-of-words view
encoder_args = {
    "text_bow": {
        "hidden_dims": [256, 128],
        "activation": "relu",
        "bias": True,
        "dropout": 0.1
    }
}

decoder_args = {
    "text_bow": {
        "hidden_dims": [128, 256],
        "activation": "relu",
        "bias": True,
        "dropout": 0.1
    }
}

# GTM trains during construction (num_steps), so there is no separate .fit() call.
# WAE prior gives stable training; update_prior=True lets the prevalence covariates
# shape the topic prior. n_topics=10 as required.
#
# NOTE: at 2000 steps the loss was still falling and the Divergence Loss was ~0.02,
# so the topics had not separated (every topic showed the same generic filler words).
# Training to 8000 steps gives the topics time to differentiate. w_prior is raised
# to 5 to put more pressure on the divergence term that pushes topics apart.
gtm = GTM(
    train_dataset,
    n_topics=10,
    ae_type="wae",
    doc_topic_prior="logistic_normal",
    update_prior=True,
    encoder_args=encoder_args,
    decoder_args=decoder_args,
    batch_size=64,
    w_prior=5,                  # stronger push on the divergence term -> more distinct topics
    return_best_model=True,
    num_steps=8000,             # 2000 was far too few; topics were still undifferentiated
    print_every_n_steps=1000,
    print_topics=True,
    print_covariates=True
)

# If topics still look mushy, keep training for more steps without rebuilding:
# gtm.num_steps = gtm.steps + 4000
# gtm.train(train_dataset)

#TODO Did the interpretability increased with respect to BERTopic?
# After sufficient training, yes. By explicitly modeling party, year, and chamber as covariates,
# GTM separates variance driven by speaker/time characteristics from the underlying topical signal.
# BERTopic relies on dense sentence embeddings that conflate semantic and stylistic variation,
# producing broader, harder-to-label clusters (note how several BERTopic topics above were
# dominated by procedural filler like "bill", "go", "other"). GTM's covariate-aware bag-of-words
# model yields crisper, more coherent word lists per topic — *provided it is trained long enough*;
# undertrained, every topic collapses to the same high-frequency words.

Step   1000	Mean Training Loss:8.4560127
Rec Loss:7.5317554
Divergence Loss:0.9242573
Pred Loss:0.0000000

Topic_0: ['tax', 'percent', 'job', 'health', 'cost']
Topic_1: ['community', 'student', 'school', 'child', 'service']
Topic_2: ['life', 'great', 'man', 'family', 'honor']
Topic_3: ['know', 'come', 'country', 'american', 'right']
Topic_4: ['know', 'friend', 'come', 'good', 'leader']
Topic_5: ['child', 'country', 'right', 'law', 'life']
Topic_6: ['provide', 'program', 'health', 'legislation', 'care']
Topic_7: ['tax', 'budget', 'know', 'american', 'come']
Topic_8: ['child', 'school', 'program', 'education', 'provide']
Topic_9: ['budget', 'know', 'come', 'majority', 'let']
Intercept: ['child', 'country', 'provide', 'legislation', 'program']
C(party)[T.Independent]: ['child', 'program', 'provide', 'health', 'care']
C(party)[T.Libertarian]: ['child', 'country', 'legislation', 'law', 'know']
C(party)[T.Popular Democrat]: ['child', 'provide', 'program', 'help', 'community']
C(party)[T.Repu

In [14]:
#TODO feed the topics (first 10 words) into the LLM of your choice (either using an API or manually) and ask it to interpret them

# get_topic_words returns a dict {topic_name: [w1, w2, ...]} for the top-K words per topic
topic_words = gtm.get_topic_words(topK=10)

# Build a prompt that can be pasted directly into any chat LLM, or sent via an API
prompt_lines = [
    "Interpret the following topics discovered from U.S. Congressional speeches (1994–2024).",
    "For each topic, provide: (1) a short label, and (2) a one-sentence description.\n"
]
for topic, words in topic_words.items():
    prompt_lines.append(f"{topic}: {', '.join(words)}")

prompt = "\n".join(prompt_lines)
print(prompt)

# --- Optional: automate via the Anthropic API ---
# import anthropic
# client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from the environment
# message = client.messages.create(
#     model="claude-sonnet-4-6",
#     max_tokens=1024,
#     messages=[{"role": "user", "content": prompt}]
# )
# print(message.content[0].text)

Interpret the following topics discovered from U.S. Congressional speeches (1994–2024).
For each topic, provide: (1) a short label, and (2) a one-sentence description.

Topic_0: water, energy, provide, use, program, legislation, project, industry, oil, cost
Topic_1: school, student, education, child, program, community, teacher, provide, high, college
Topic_2: life, know, man, day, nation, great, country, family, american, come
Topic_3: right, american, country, policy, democracy, government, law, world, use, know
Topic_4: team, championship, game, win, know, coach, season, great, record, player
Topic_5: woman, child, violence, law, crime, life, right, gun, abortion, provide
Topic_6: law, legislation, business, require, consumer, provision, provide, use, company, insurance
Topic_7: tax, budget, cut, pay, percent, deficit, family, debt, american, income
Topic_8: health, care, insurance, cost, coverage, patient, provide, plan, program, legislation
Topic_9: process, budget, majority, repu

In [15]:
#TODO Finally, do Republicans use different words by topic? Show the 10 most used words by topic for Republicans

# Because party is a *content* covariate, GTM lets us condition the topic-word distribution
# on a specific party — this directly shows how word usage shifts for Republicans.

# Show the exact content covariate labels the model exposes (patsy-generated names)
print("Available content covariates:", gtm.content_colnames)

# Find the Republican label automatically (e.g. "C(party)[T.Republican]") so we don't
# have to hard-code patsy's naming, which depends on the reference category
rep_label = [c for c in gtm.content_colnames if "Republican" in c][0]

# Baseline ("Intercept") + the Republican shift gives the Republican-conditioned word lists
rep_topic_words = gtm.get_topic_words(
    l_content_covariates=["Intercept", rep_label],
    topK=10
)

print(f"\nTop 10 words per topic, conditioned on Republican ({rep_label}):\n")
import pandas as pd
print(pd.DataFrame(rep_topic_words).T)

# For comparison, the party-agnostic baseline topics:
# print(pd.DataFrame(gtm.get_topic_words(l_content_covariates=["Intercept"], topK=10)).T)
# Differences between the two tables answer the question: yes, Republicans emphasize
# different vocabulary within the same latent topics.

Available content covariates: ['Intercept', 'C(party)[T.Independent]', 'C(party)[T.Libertarian]', 'C(party)[T.Popular Democrat]', 'C(party)[T.Republican]', 'C(year)[T.1995]', 'C(year)[T.1996]', 'C(year)[T.1997]', 'C(year)[T.1998]', 'C(year)[T.1999]', 'C(year)[T.2000]', 'C(year)[T.2001]', 'C(year)[T.2002]', 'C(year)[T.2003]', 'C(year)[T.2004]', 'C(year)[T.2005]', 'C(year)[T.2006]', 'C(year)[T.2007]', 'C(year)[T.2008]', 'C(year)[T.2009]', 'C(year)[T.2010]', 'C(year)[T.2011]', 'C(year)[T.2012]', 'C(year)[T.2013]', 'C(year)[T.2014]', 'C(year)[T.2015]', 'C(year)[T.2016]', 'C(year)[T.2017]', 'C(year)[T.2018]', 'C(year)[T.2019]', 'C(year)[T.2020]', 'C(year)[T.2021]', 'C(year)[T.2022]', 'C(year)[T.2023]', 'C(year)[T.2024]', 'C(chamber)[T.Senate]']

Top 10 words per topic, conditioned on Republican (C(party)[T.Republican]):

                0         1             2              3        4           5  \
Topic_0     water    energy           use        percent  provide     program   
Topic_1   

In [17]:
#TODO Finally, do Democrats use different words by topic? Show the 10 most used words by topic for Democrats

# Because party is a *content* covariate, GTM lets us condition the topic-word distribution
# on a specific party — this directly shows how word usage shifts for Democrats.

# Show the exact content covariate labels the model exposes (patsy-generated names)
print("Available content covariates:", gtm.content_colnames)

# Find the plain "Democrat" label, explicitly EXCLUDING "Popular Democrat".
# Patsy drops the reference category: in this model "Democrat" is the baseline, so there is
# no "C(party)[T.Democrat]" term — the Intercept alone already represents Democrats.
dem_labels = [c for c in gtm.content_colnames if "Democrat" in c and "Popular" not in c]

if dem_labels:
    covariates = ["Intercept", dem_labels[0]]
    label_desc = dem_labels[0]
else:
    # "Democrat" is the reference category, absorbed into the Intercept
    covariates = ["Intercept"]
    label_desc = "Intercept (Democrat is the reference category)"

# Condition the topic-word distribution on Democrats
dem_topic_words = gtm.get_topic_words(
    l_content_covariates=covariates,
    topK=10
)

print(f"\nTop 10 words per topic, conditioned on Democrat ({label_desc}):\n")
import pandas as pd
print(pd.DataFrame(dem_topic_words).T)

# For comparison, the Republican-conditioned topics:
# rep_label = [c for c in gtm.content_colnames if "Republican" in c][0]
# print(pd.DataFrame(gtm.get_topic_words(l_content_covariates=["Intercept", rep_label], topK=10)).T)
# Differences between the two tables answer the question: yes, Democrats emphasize
# different vocabulary within the same latent topics.

Available content covariates: ['Intercept', 'C(party)[T.Independent]', 'C(party)[T.Libertarian]', 'C(party)[T.Popular Democrat]', 'C(party)[T.Republican]', 'C(year)[T.1995]', 'C(year)[T.1996]', 'C(year)[T.1997]', 'C(year)[T.1998]', 'C(year)[T.1999]', 'C(year)[T.2000]', 'C(year)[T.2001]', 'C(year)[T.2002]', 'C(year)[T.2003]', 'C(year)[T.2004]', 'C(year)[T.2005]', 'C(year)[T.2006]', 'C(year)[T.2007]', 'C(year)[T.2008]', 'C(year)[T.2009]', 'C(year)[T.2010]', 'C(year)[T.2011]', 'C(year)[T.2012]', 'C(year)[T.2013]', 'C(year)[T.2014]', 'C(year)[T.2015]', 'C(year)[T.2016]', 'C(year)[T.2017]', 'C(year)[T.2018]', 'C(year)[T.2019]', 'C(year)[T.2020]', 'C(year)[T.2021]', 'C(year)[T.2022]', 'C(year)[T.2023]', 'C(year)[T.2024]', 'C(chamber)[T.Senate]']

Top 10 words per topic, conditioned on Democrat (Intercept (Democrat is the reference category)):

              0             1          2           3         4              5  \
Topic_0   water       program    provide      energy   project    leg